# lm_forge — Colab Nomad Session

This notebook is **stateless** — clone, pull checkpoint, train, push, die. Safe to kill at any time.

**One-time setup (do this before your first training session):**
1. Add `HF_TOKEN` in **Secrets** (key icon, left sidebar)
2. Set `GITHUB_REPO` and `HF_USERNAME` below
3. Run the **Pre-tokenize** cell once — never again

**Every session after that:** run cells 1→5 only.

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────
GITHUB_REPO  = 'https://github.com/YOUR_USERNAME/lm_forge.git'  # ← change
HF_USERNAME  = 'YOUR_HF_USERNAME'                               # ← change
EXPERIMENT   = 'exp_001_gqa_rope'                               # ← change
# ──────────────────────────────────────────────────────────────────────────

In [ ]:
# ── 1. Install deps ────────────────────────────────────────────────────────
# datasets + transformers are usually pre-installed on Colab — check first
import importlib
missing = [p for p in ['huggingface_hub','safetensors','pyyaml','datasets','transformers']
           if importlib.util.find_spec(p) is None]
if missing:
    !pip install -q {' '.join(missing)}
print('Deps OK')

In [ ]:
# ── 2. Auth HF Hub ─────────────────────────────────────────────────────────
from google.colab import userdata
import os
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print('HF_TOKEN set ✓')

In [ ]:
# ── 3. Clone / refresh repo ────────────────────────────────────────────────
import os
if not os.path.exists('lm_forge'):
    !git clone {GITHUB_REPO} lm_forge
else:
    !cd lm_forge && git pull
%cd lm_forge

In [ ]:
# ── 4. GPU check ───────────────────────────────────────────────────────────
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

In [ ]:
# ── 5. TRAIN (pulls latest checkpoint from Hub automatically) ──────────────
!python experiments/{EXPERIMENT}/train.py

---
## One-time setup — pre-tokenize your dataset

Run this cell **once** before your first training session.
It tokenizes TinyStories (~470MB) and pushes the result to your Hub.
All future sessions pull the tokenized data in seconds.

Takes ~10 minutes on Colab. After this, update `data_hub_repo` in `config.yaml`.

In [ ]:
# ── ONE-TIME: Pre-tokenize & push to Hub ──────────────────────────────────
DATA_HUB_REPO = f'{HF_USERNAME}/tinystories-gpt2'

!python -m engine.data.pretokenize \
    --dataset roneneldan/TinyStories \
    --tokenizer gpt2 \
    --hub_repo {DATA_HUB_REPO} \
    --output_dir data/tinystories_gpt2 \
    --num_proc 2

print(f'\nDone! Now set in config.yaml:')
print(f'  training:')
print(f'    data_hub_repo: "{DATA_HUB_REPO}"')
print(f'    vocab_size: 50257  # in the model section')

---
## Manual / diagnostic cells

In [ ]:
# ── CPU smoke test (no GPU, no real data, fast) ────────────────────────────
# !python experiments/{EXPERIMENT}/train.py --cpu --steps 30 --synthetic

In [ ]:
# ── Benchmark: MFU + throughput ────────────────────────────────────────────
import sys; sys.path.insert(0, '.')
from engine import CausalLM, load_experiment_config
from engine.training import DeviceManager
from engine.utils import Profiler

exp   = load_experiment_config(f'experiments/{EXPERIMENT}/config.yaml')
dm    = DeviceManager(exp.training)
model = dm.prepare(CausalLM(exp.model))

results = Profiler.benchmark(model, dm, seq_len=exp.training.seq_len,
                             batch_size=exp.training.batch_size, n_steps=30)
for k, v in results.items():
    print(f'  {k:<22}: {v}')

In [ ]:
# ── Inspect tokenized dataset ──────────────────────────────────────────────
from engine.data import MemmapDataset

ds = MemmapDataset.from_hub(f'{HF_USERNAME}/tinystories-gpt2',
                            split='train', seq_len=512)
print(ds)
item = ds[0]
print('input_ids shape:', item['input_ids'].shape)
print('first 10 tokens:', item['input_ids'][:10].tolist())

In [ ]:
# ── List available component types ─────────────────────────────────────────
from engine import list_attention_types, list_pe_types, list_ffn_types
print('Attention:', list_attention_types())
print('PE:       ', list_pe_types())
print('FFN:      ', list_ffn_types())

In [ ]:
# ── Run ablation sweep ─────────────────────────────────────────────────────
# Runs 8 variants (attention × PE × FFN), ~4h total on T4
# !python experiments/ablation_001/run_ablation.py --steps 500

In [ ]:
# ── Generate model card + push to Hub ──────────────────────────────────────
# from engine import load_experiment_config
# from engine.utils import generate_model_card
# from engine.models import HFCausalLM, CausalLM
#
# exp   = load_experiment_config(f'experiments/{EXPERIMENT}/config.yaml')
# model = CausalLM.from_pretrained(f'checkpoints/{exp.name}/final')
#
# generate_model_card(exp, 'checkpoints/README.md',
#                     author=HF_USERNAME,
#                     eval_results={'eval_loss': 2.31, 'perplexity': 10.1})
#
# hf_model = HFCausalLM.from_engine_model(model)
# hf_model.push_to_hub(exp.hub.repo_id)

In [ ]:
# ── LoRA fine-tuning (after pip install peft) ──────────────────────────────
# from peft import LoraConfig, get_peft_model, TaskType
# from engine.models import HFCausalLM
# from engine.config import LMForgeConfig
#
# exp    = load_experiment_config(f'experiments/{EXPERIMENT}/config.yaml')
# hf_cfg = LMForgeConfig.from_model_config(exp.model)
# model  = HFCausalLM.from_pretrained(exp.hub.repo_id)
#
# lora_cfg = LoraConfig(
#     task_type=TaskType.CAUSAL_LM,
#     r=16, lora_alpha=32,
#     target_modules=['q_proj', 'v_proj'],  # LLaMA names — works out of the box
#     lora_dropout=0.05,
# )
# model = get_peft_model(model, lora_cfg)
# model.print_trainable_parameters()